# Qianfan-OCR on the Dutch receipt corpus (v2 Appendix A, step 3)

Runs `baidu/Qianfan-OCR` over every receipt photo and writes `<sha256-of-image-bytes>.qianfan.json`:

```json
{ "modelRevision": "<HF commit>", "markdown": "…", "kie": "<raw string>" }
```

Copy the files next to the truth files in `fixtures/receipts-nl/`, then run `pnpm eval:extraction`.

**Runtime:** GPU, L4 or A100 (bf16). On a T4 set `DTYPE = torch.float16` in the model cell.
Decoding is greedy (`do_sample=False`) and the two prompts below are fixed: they are the same strings as
`QIANFAN_PROMPTS` in `packages/receipt-nl`.

In [ ]:
!pip -q install -U transformers accelerate pillow huggingface_hub

In [ ]:
import hashlib, json, pathlib, torch
from huggingface_hub import HfApi
from PIL import Image
from transformers import AutoModelForImageTextToText, AutoProcessor

MODEL_PATH = "baidu/Qianfan-OCR"
DTYPE = torch.bfloat16            # T4: torch.float16

# Fixed prompts (v2 Appendix A). Do not edit: receipt-nl's parser expects this exact KIE shape.
PROMPT_MARKDOWN = "Parse this document to Markdown."
PROMPT_KIE = "Extract from this Dutch receipt as JSON: merchant_name, kvk_number, btw_number, postcode, date, time, lines[{description, qty, unit_price, total, btw_rate}], btw_breakdown[{rate, base, btw}], total, rounding, payment_method. Use null for missing fields. Copy numbers exactly as printed."

# The commit that was actually loaded, so a recording can be reproduced.
MODEL_REVISION = HfApi().model_info(MODEL_PATH).sha

model = AutoModelForImageTextToText.from_pretrained(
    MODEL_PATH, torch_dtype=DTYPE, device_map="auto"
).eval()
processor = AutoProcessor.from_pretrained(MODEL_PATH)
print("revision:", MODEL_REVISION)

In [ ]:
def ask(image, prompt, max_new_tokens=2048):
    messages = [{"role": "user", "content": [
        {"type": "image", "image": image},
        {"type": "text", "text": prompt},
    ]}]
    inputs = processor.apply_chat_template(
        messages, add_generation_prompt=True, tokenize=True, return_dict=True, return_tensors="pt"
    ).to(model.device)
    with torch.no_grad():
        output_ids = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
    generated = output_ids[:, inputs["input_ids"].shape[1]:]
    return processor.batch_decode(generated, skip_special_tokens=True)[0]

## Receipt photos

Upload the photos (`<id>.jpg`, exactly the files that are in `fixtures/receipts-nl/`: the file name does not matter,
the SHA-256 of the bytes does) or mount Drive and set `IMAGE_DIR`.

In [ ]:
from google.colab import files

IMAGE_DIR = pathlib.Path("receipts")
IMAGE_DIR.mkdir(exist_ok=True)
for name, data in files.upload().items():
    (IMAGE_DIR / name).write_bytes(data)
print(sorted(p.name for p in IMAGE_DIR.iterdir()))

In [ ]:
OUT_DIR = pathlib.Path("qianfan-out")
OUT_DIR.mkdir(exist_ok=True)

for path in sorted(IMAGE_DIR.iterdir()):
    if path.suffix.lower() not in {".jpg", ".jpeg", ".png", ".webp"}:
        continue
    raw = path.read_bytes()
    sha256 = hashlib.sha256(raw).hexdigest()          # the name the recorded extractor looks up
    image = Image.open(path).convert("RGB")
    record = {
        "modelRevision": MODEL_REVISION,
        "markdown": ask(image, PROMPT_MARKDOWN),
        "kie": ask(image, PROMPT_KIE),                # kept as the raw string: parsing is receipt-nl's job
    }
    (OUT_DIR / f"{sha256}.qianfan.json").write_text(json.dumps(record, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")
    print(path.name, "->", f"{sha256}.qianfan.json")

In [ ]:
!zip -qr qianfan-out.zip qianfan-out
files.download("qianfan-out.zip")